# 02_train_baselines
Trains baseline models by fold, caches predictions/metrics/models, and skips unavailable optional models.

In [ ]:
from pathlib import Path
import sys
import traceback

import pandas as pd
import yaml

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data import load_dataset
from src.pipeline import run_baseline_fold
from src.splits import load_splits

with open(ROOT / "configs" / "baselines.yaml", "r", encoding="utf-8") as f:
    baseline_cfg = yaml.safe_load(f)

bundle = load_dataset(
    csv_path=ROOT / "data" / "all_data_clean.csv",
    target_col="sbp_binary",
    subject_col="PAT_ID",
)
splits = load_splits(ROOT / "artifacts" / "splits" / "outer_splits.json")

results = []
for model_name in baseline_cfg["baselines"]["enabled"]:
    for fold_info in splits:
        fold = fold_info["fold"]
        try:
            result = run_baseline_fold(
                df=bundle.df,
                feature_cols=bundle.feature_cols,
                target_col=bundle.target_col,
                subject_col=bundle.subject_col,
                train_idx=fold_info["train_idx"],
                test_idx=fold_info["test_idx"],
                model_name=model_name,
                fold=fold,
                out_dir=ROOT / "artifacts" / "metrics",
            )
            result["status"] = "ok"
            results.append(result)
            print(f"{model_name} fold {fold}: ok")
        except Exception as exc:
            results.append({"model": model_name, "fold": fold, "status": "skipped", "error": str(exc)})
            print(f"{model_name} fold {fold}: skipped")
            print(traceback.format_exc(limit=1))

pd.DataFrame(results).head()